# 📊 Prompt Diff & Semantic Drift Visualizer
**Series: 2/4** **Author:** Master Timo  
**Stack:** 100% Local (Ollama + `nomic-embed-text` + `rich` text rendering)

### The Problem
When you refine a prompt template, evaluating it against a single test case manually is a trap. A change that fixes Case A might completely break formatting or context nuance in Case B. Furthermore, look-and-feel changes (lexical differences) don't always equal a shift in core meaning (semantic drift).

### The Engineering Solution
This notebook builds a scalable, local testing harness that evaluates prompt alterations across a multi-case dataset. It measures:
1. **Structural Text Deltas:** Token count adjustments and line-by-line insertions/deletions.
2. **Mathematical Semantic Drift:** Maps outputs into vector space using `nomic-embed-text` and calculates Angular Distance derived from Cosine Similarity:
   $$\text{Semantic Drift} = \frac{\arccos(\text{Cosine Similarity})}{\pi}$$
   *Why this formula?* Raw cosine similarity often groups tightly between 0.85 and 0.98 for LLM outputs. Transforming it into normalized distance gives a much cleaner linear visualization of how far the text drifted.

### Pre-requisites
```bash
ollama pull nomic-embed-text
pip install requests rich

### Code Implementation

In [2]:
%pip install requests rich

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [10]:
import requests
import json
import difflib
import math
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.columns import Columns

# Initialize Rich Console for beautiful notebook UI rendering
console = Console()

OLLAMA_URL = "http://localhost:11434"
GENERATION_MODEL = "mistral"
EMBEDDING_MODEL = "nomic-embed-text"


# ---------------------------------------------------------
# 🛠️ SYSTEM HEALTH & DEPENDENCY CHECK
# ---------------------------------------------------------
try:
    health = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    models = [m["name"] for m in health.json().get("models", [])]
    
    print(f"✅ Ollama Local Engine is active.")
    print(f"📦 Available System Models: {models}")
    
    # Verify exact requirements exist in the local tag manifest
    if not any(GENERATION_MODEL in m for m in models):
        print(f"⚠️  Warning: '{GENERATION_MODEL}' not found. Run: ollama pull {GENERATION_MODEL}")
    if not any(EMBEDDING_MODEL in m for m in models):
        print(f"⚠️  Warning: '{EMBEDDING_MODEL}' not found. Run: ollama pull {EMBEDDING_MODEL}")
        
except Exception as e:
    print(f"❌ Ollama Environment Error: {e}\n👉 Please execute 'ollama serve' in your terminal daemon.")

✅ Ollama Local Engine is active.
📦 Available System Models: ['llama3.1:8b', 'llava:13b', 'nomic-embed-text:latest', 'gemma3:latest', 'mistral:latest']


In [4]:
def run_generation(prompt: str, temperature: float = 0.2) -> str:
    """Query local LLM via Ollama."""
    payload = {
        "model": GENERATION_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature}
    }
    try:
        response = requests.post(f"{OLLAMA_URL}/api/generate", json=payload)
        response.raise_for_status()
        return response.json()["response"].strip()
    except Exception as e:
        return f"Generation Error: {e}"

In [5]:

def get_embedding(text: str) -> list:
    """Generate vector embeddings locally."""
    payload = {"model": EMBEDDING_MODEL, "prompt": text}
    try:
        response = requests.post(f"{OLLAMA_URL}/api/embeddings", json=payload)
        response.raise_for_status()
        return response.json()["embedding"]
    except Exception:
        return [0.0] * 768 # Fallback empty vector dimensions


In [6]:

def compute_semantic_drift(vec_a: list, vec_b: list) -> float:
    """Calculate normalized angular distance/drift from cosine similarity."""
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    mag_a = math.sqrt(sum(a * a for a in vec_a))
    mag_b = math.sqrt(sum(b * b for b in vec_b))
    
    if not mag_a or not mag_b:
        return 1.0
    
    cosine_sim = max(min(dot_product / (mag_a * mag_b), 1.0), -1.0)
    # Convert similarity to angular distance scale [0, 1]
    angular_distance = math.acos(cosine_sim) / math.pi
    return angular_distance


In [8]:

def render_visual_diff(text_a: str, text_b: str) -> str:
    """Create a formatted, colorized line-by-line text delta visualization string."""
    diff = difflib.ndiff(text_a.splitlines(), text_b.splitlines())
    visual_lines = []
    for line in diff:
        if line.startswith('+ '):
            visual_lines.append(f"[green]{line}[/green]")
        elif line.startswith('- '):
            visual_lines.append(f"[red]{line}[/red]")
    return "\n".join(visual_lines) if visual_lines else "[dim]No structural changes (Identical lines)[/dim]"


In [7]:

def execute_diff_harness(dataset: list, baseline_tmpl: str, optimized_tmpl: str):
    """Iterate through dataset, execute prompt versions, compute drift, render matrix."""
    
    # Global Summary Matrix Table
    summary_table = Table(title="📊 Prompt Migration Test Suite Matrix", title_style="bold cyan")
    summary_table.add_column("ID", style="dim", width=4)
    summary_table.add_column("Input Fragment (Truncated)", width=30)
    summary_table.add_column("Δ Length (Tokens/Chars)", justify="right")
    summary_table.add_column("Semantic Drift Score", justify="center")
    summary_table.add_column("Impact Classification", justify="left")

    console.print(Panel("[bold green]🚀 Launching Multi-Scenario Prompt Diff Pipeline[/bold green]\nRunning production scenarios against Ollama stack...", border_style="green"))

    for item in dataset:
        case_id = item["id"]
        input_val = item["input_data"]
        
        p1 = baseline_tmpl.format(input_data=input_val)
        p2 = optimized_tmpl.format(input_data=input_val)
        
        # Execute generations
        out_1 = run_generation(p1)
        out_2 = run_generation(p2)
        
        # Length delta tracking
        len_delta = len(out_2) - len(out_1)
        len_prefix = "+" if len_delta >= 0 else ""
        
        # Semantic vector computation
        v1 = get_embedding(out_1)
        v2 = get_embedding(out_2)
        drift_score = compute_semantic_drift(v1, v2)
        
        # Define classification threshold badges
        if drift_score < 0.03:
            classification = "[bright_blue]Cosmetic Changes Only[/bright_blue]"
        elif drift_score < 0.12:
            classification = "[yellow]Moderate Structural Rephrase[/yellow]"
        else:
            classification = "[bold red]High Semantic Shift / Output Split[/bold red]"
            
        summary_table.add_row(
            str(case_id), 
            input_val[:28] + "...", 
            f"{len_prefix}{len_delta} chars", 
            f"{drift_score:.4f}", 
            classification
        )
        
        # Detailed Individual Breakdown View
        diff_view = render_visual_diff(out_1, out_2)
        
        console.print(f"\n[bold yellow]🔍 Deep-Dive Breakdown: Case #{case_id}[/bold yellow]")
        console.print(f"[dim]Input Context:[/dim] {input_val}")
        
        panel_p1 = Panel(out_1, title="🔴 Baseline Output (P1)", border_style="red", expand=True)
        panel_p2 = Panel(out_2, title="&gamma; Optimized Output (P2)", border_style="green", expand=True)
        panel_diff = Panel(diff_view, title="🛠️ Text Line Changes (Diff)", border_style="cyan", expand=True)
        
        # Render side-by-side structures using rich columns
        console.print(Columns([panel_p1, panel_p2]))
        console.print(panel_diff)
        console.print("═" * 90)

    # Render global output analytics layout matrix
    console.print("\n")
    console.print(summary_table)


In [ ]:

# ---------------------------------------------------------
# Production Multi-Example Evaluation Dataset
# ---------------------------------------------------------
production_dataset = [
    {
        "id": 1,
        "scenario": "Database Performance",
        "input_data": "SQL query execution performance suddenly spiked to 12000ms on the user sessions table index join."
    },
    {
        "id": 2,
        "scenario": "Security/Auth Vulnerability",
        "input_data": "The front-end authorization framework token cookie is missing its HttpOnly flag, showing security warnings."
    },
    {
        "id": 3,
        "scenario": "DevOps Pipeline Failure",
        "input_data": "Our pipeline container runner hit an out-of-memory SIGKILL exception during the Webpack application compilation layer."
    }
]

# Prompt Configurations to Test (Targeting System Response Engineering)
naive_prompt_template = "Summarize this log: {input_data}"

engineered_prompt_template = """Analyze this infrastructure incident update log. 
Output a rigid 1-line engineering summary. Then list exactly two technical blast-radius risks.
No conversational preambles.

Log Entry: {input_data}"""

# Execute the framework
execute_diff_harness(production_dataset, naive_prompt_template, engineered_prompt_template)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 Launching Multi-Scenario Prompt Diff Pipeline                                                                │
│ Running production scenarios against Ollama stack...                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🔍 Deep-Dive Breakdown: Case #1

Input Context: SQL query execution performance suddenly spiked to 12000ms on the user sessions table index join.

╭──────────────────────────────────────────── 🔴 Baseline Output (P1) ────────────────────────────────────────────╮
│ The log indicates a significant increase in SQL query execution time for operations involving the user sessions │
│ table's index join. Specifically, the query execution time has spiked to approximately 12,000 milliseconds (or  │
│ 12 seconds). This sudden performance issue might be due to factors such as increased data volume, inefficient   │
│ queries, or index corruption, and may require further investigation and optimization to improve query           │
│ performance.                                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭───────────────────────────────────────── &gamma; Optimized Output (P2) ─────────────────────────────────────────╮
│ SQL User Sessions Table Index Join Performance Anomaly: Sudden spike in query execution time (12000ms)          │
│ indicates potential database bottleneck or index corruption, which could lead to increased blast radius for     │
│ subsequent queries and potential system instability.                                                            │
│                                                                                                                 │
│ Technical Blast-Radius Risks:                                                                                   │
│ 1. Query Propagation: Affected query may propagate to other dependent processes, causing performance            │
│ degradation across the system.                                                                                  │
│ 2. Data Integrity Issues: Corrupted index could lead to data inconsistencies and potential cascading failures   │
│ in related operations.                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────── 🛠️ Text Line Changes (Diff) ───────────────────────────────────────────╮
│ - The log indicates a significant increase in SQL query execution time for operations involving the user        │
│ sessions table's index join. Specifically, the query execution time has spiked to approximately 12,000          │
│ milliseconds (or 12 seconds). This sudden performance issue might be due to factors such as increased data      │
│ volume, inefficient queries, or index corruption, and may require further investigation and optimization to     │
│ improve query performance.                                                                                      │
│ + SQL User Sessions Table Index Join Performance Anomaly: Sudden spike in query execution time (12000ms)        │
│ indicates potential database bottleneck or index corruption, which could lead to increased blast radius for     │
│ subsequent queries and potential system instability.                                                            │
│ +                                                                                                               │
│ + Technical Blast-Radius Risks:                                                                                 │
│ + 1. Query Propagation: Affected query may propagate to other dependent processes, causing performance          │
│ degradation across the system.                                                                                  │
│ + 2. Data Integrity Issues: Corrupted index could lead to data inconsistencies and potential cascading failures │
│ in related operations.                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Input Context: Our pipeline container runner hit an out-of-memory SIGKILL exception during the Webpack application 
compilation layer.

╭──────────────────────────────────────────── 🔴 Baseline Output (P1) ────────────────────────────────────────────╮
│ The log indicates that a container running in a pipeline encountered an Out-of-Memory error (SIGKILL exception) │
│ while compiling a Webpack application. This suggests that the application's memory requirements exceeded the    │
│ allocated resources, causing the system to terminate the process.                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭───────────────────────────────────────── &gamma; Optimized Output (P2) ─────────────────────────────────────────╮
│ Pipeline container failure due to OOM SIGKILL during Webpack application compilation; potential risks include   │
│ data loss and unintended system instability within immediate container environment, as well as possible         │
│ cascading effects on adjacent containers in the pipeline.                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────── 🛠️ Text Line Changes (Diff) ───────────────────────────────────────────╮
│ - The log indicates that a container running in a pipeline encountered an Out-of-Memory error (SIGKILL          │
│ exception) while compiling a Webpack application. This suggests that the application's memory requirements      │
│ exceeded the allocated resources, causing the system to terminate the process.                                  │
│ + Pipeline container failure due to OOM SIGKILL during Webpack application compilation; potential risks include │
│ data loss and unintended system instability within immediate container environment, as well as possible         │
│ cascading effects on adjacent containers in the pipeline.                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

══════════════════════════════════════════════════════════════════════════════════════════

                                       📊 Prompt Migration Test Suite Matrix                                       
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID   ┃ Input Fragment (Truncated)     ┃ Δ Length (Tokens/Chars) ┃ Semantic Drift Score ┃ Impact Classification  ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1    │ SQL query execution            │              +112 chars │        0.1730        │ High Semantic Shift /  │
│      │ performa...                    │                         │                      │ Output Split           │
│ 2    │ The front-end authorization    │               -93 chars │        0.1983        │ High Semantic Shift /  │
│      │ ...                            │                         │                      │ Output Split           │
│ 3    │ Our pipeline container         │               -15 chars │        0.1846        │ High Semantic Shift /  │
│      │ runne...                       │                         │                      │ Output Split           │
└──────┴────────────────────────────────┴─────────────────────────┴──────────────────────┴────────────────────────┘

## 🏁 Module 02 Summary Recap

| Evaluated Metric Layer | Engineering Purpose | Local Tool Mechanism |
| :--- | :--- | :--- |
| **Lexical Delta Tracking** | Identifies absolute text line changes, shifts, and insertions. | Python `difflib.ndiff` |
| **Vector Vector Space Mapping** | Measures changes in contextual intent rather than just words. | `nomic-embed-text` |
| **Angular Drift Calculation** | Normalizes raw cosine variance into a scannable linear scale ($[0,1]$). | Distance Mapping Formula |